In [ ]:
# --- Project bootstrap -------------------------------------------------------
# This notebook lives in notebooks/, but every path below is written relative to
# the project root (data/..., experiments/...), and the packages it imports
# (nn, training, evaluation, experiments, data) sit at that root as well.
#
# Locating the root by marker rather than by a fixed "../" keeps the notebook
# correct whether Jupyter was started inside notebooks/ or at the project root.
import os
import sys
from pathlib import Path


def _find_project_root():
    """Walk upward until a directory holding both nn/ and training/ is found."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "nn").is_dir() and (candidate / "training").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the project root: no parent of "
        f"{Path.cwd()} contains both nn/ and training/."
    )


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {Path.cwd()}")


In [ ]:
import os
import json
import copy
import numpy as np

from data.data_handler.data_loader import load_monk
from training.model_factory import build_model_from_cfg
from training.trainer import Trainer
from nn.metrics import MSE as MSEMetric

from evaluation.ensemble_utils import (
    stack_histories,
    predict_proba,
    majority_vote,
    accuracy,
    bce_loss,
    mean_std,
    plot_runs_with_mean,
    compute_best_epoch_from_kfold,
)

SEEDS = [9, 11, 35, 42, 51, 68, 74, 81, 99]

In [ ]:
# ============================================================
# MONK1 — best_epoch from 5-fold CV (TRAIN ONLY) + multi-seed retrain + Loss/Acc/MSE plots
# ============================================================

NAME = "MONK1"
best_cfg_path = "experiments/MONK/results/monk1/best_config.json"
train_path    = "data/MONK/MONK1/monks-1.train"
test_path     = "data/MONK/MONK1/monks-1.test"

print("\n" + "#" * 90)
print(f"### {NAME} — {len(SEEDS)} seeds + majority vote ensemble")
print("#" * 90)

# Load cfg
if not os.path.exists(best_cfg_path):
    raise FileNotFoundError(f"[{NAME}] best_config not found: {best_cfg_path}")
with open(best_cfg_path, "r") as f:
    cfg_raw = json.load(f)

# The test set is passed to fit() as X_val below purely so the Train-vs-Test
# curves can be logged. Early stopping must therefore be switched off: with
# restore_best_weights it would otherwise pick the epoch that happens to look
# best on the test set, which is model selection on the test set.
cfg = copy.deepcopy(cfg_raw)
cfg.setdefault("callbacks", {})
cfg["callbacks"]["early_stopping"] = False

# Load data
if not os.path.exists(train_path):
    raise FileNotFoundError(f"[{NAME}] Train file not found: {train_path}")
if not os.path.exists(test_path):
    raise FileNotFoundError(f"[{NAME}] Test file not found:  {test_path}")

print("Train file:", train_path)
print("Test  file:", test_path)

X_train, y_train = load_monk(train_path, encode=True)
X_test,  y_test  = load_monk(test_path,  encode=True)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)

# -------------------------
# Training params
# -------------------------
epochs = int(cfg.get("training", {}).get("epochs", 1000))
batch_size = int(cfg.get("training", {}).get("batch_size", 16))
shuffle = bool(cfg.get("training", {}).get("shuffle", True))
drop_last = bool(cfg.get("training", {}).get("drop_last", False))
include_reg_in_val = bool(cfg.get("training", {}).get("include_reg_in_val", False))

# -------------------------
# 1) Best epoch from 5-fold CV on TRAIN only (no test leakage)
#    - compute per seed, then average across seeds
# -------------------------
cv_cfg = cfg.get("cv", {}) or {}
K = int(cv_cfg.get("k", 5))

best_epoch_means = []
best_epoch_stds = []

print("\n" + "#" * 90)
print(f"[{NAME}] Computing best_epoch from {K}-Fold CV on TRAIN ONLY (per seed, then mean)")
print("#" * 90)

for s in SEEDS:
    be_mean, be_std, be_list = compute_best_epoch_from_kfold(
        X_train=X_train,
        y_train=y_train,
        cfg=cfg,
        seed=int(s),
        build_model_from_cfg_fn=build_model_from_cfg,
        trainer_cls=Trainer,
        k=K,
        monitor="val_loss",
    )
    if be_mean is None:
        print(f"  seed={s}: (no best epoch found)")
        continue

    best_epoch_means.append(be_mean)
    best_epoch_stds.append(be_std)
    print(f"  seed={s}: best_epoch_mean={be_mean:.2f} (std across folds={be_std:.2f}) | folds={be_list}")

if len(best_epoch_means) == 0:
    raise RuntimeError("Could not compute best_epoch from k-fold histories (missing 'val_loss'?).")

best_epoch_global = int(np.round(float(np.mean(best_epoch_means))))
print("\n" + "-" * 90)
print(f"[{NAME}] GLOBAL best_epoch (mean across seeds of k-fold mean): {best_epoch_global}  (0-based)")
print("-" * 90 + "\n")


# -------------------------
# 2) Full retraining on TRAIN and evaluation on TEST (one model per seed)
# -------------------------
all_histories, all_models, all_evals, run_lengths = [], [], [], []

print("\n" + "#" * 90)
print(f"[{NAME}] Full retraining for {len(SEEDS)} seeds + logging Train vs Test curves")
print("#" * 90)

for s in SEEDS:
    print("\n" + "=" * 70)
    print(f"[{NAME}] RUN seed={s}")
    print("=" * 70)

    np.random.seed(int(s))

    model = build_model_from_cfg(
        run_cfg=cfg,
        seed=int(s),
        in_dim=X_train.shape[1],
        out_dim=y_train.shape[1],
        task="binary",
    )

    # ---- add MSE metric so History includes MSE / val_MSE
    if not any(m.__class__.__name__ == "MSE" for m in getattr(model, "metrics", [])):
        model.metrics.append(MSEMetric())

    trainer = Trainer(model, verbose=1)

    history = trainer.fit(
        X_train, y_train,
        X_val=X_test, y_val=y_test,   # used ONLY for plotting curves (NOT for best_epoch)
        epochs=epochs,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        seed=int(s),
        include_reg_in_val=include_reg_in_val,
    )

    hist = history.to_dict()
    all_histories.append(hist)
    all_models.append(model)

    L = len(hist.get("loss", []))
    run_lengths.append(L)
    print(f"Done. epochs_ran={L}")

    train_eval = trainer.evaluate(X_train, y_train, batch_size=256, include_regularization=True)
    test_eval  = trainer.evaluate(X_test,  y_test,  batch_size=256, include_regularization=True)

    all_evals.append({
        "seed": int(s),
        "train_loss": float(train_eval.get("loss", np.nan)),
        "train_acc":  float(train_eval.get("Accuracy", np.nan)),
        "train_mse":  float(train_eval.get("MSE", np.nan)),
        "test_loss":  float(test_eval.get("loss", np.nan)),
        "test_acc":   float(test_eval.get("Accuracy", np.nan)),
        "test_mse":   float(test_eval.get("MSE", np.nan)),
    })

print("\nEpochs per run:", run_lengths)
print("Min epochs:", min(run_lengths), "| Max epochs:", max(run_lengths))


# -------------------------
# 3) Majority-vote ensemble (final numbers)
# -------------------------
P_train = np.stack([predict_proba(m, X_train) for m in all_models], axis=0)
P_test  = np.stack([predict_proba(m, X_test)  for m in all_models], axis=0)

ytr = y_train.reshape(-1).astype(int)
yte = y_test.reshape(-1).astype(int)

yhat_tr, pmean_tr, _ = majority_vote(P_train, threshold=0.5)
yhat_te, pmean_te, _ = majority_vote(P_test,  threshold=0.5)

ens_train_acc  = accuracy(ytr, yhat_tr)
ens_test_acc   = accuracy(yte, yhat_te)
ens_train_loss = bce_loss(ytr, pmean_tr)
ens_test_loss  = bce_loss(yte, pmean_te)

print("\n================ ENSEMBLE (MAJORITY VOTE) ================")
print(f"Models: {len(all_models)}")
print(f"TRAIN BCE(mean-proba): {ens_train_loss:.6f} | acc: {ens_train_acc:.6f}")
print(f"TEST  BCE(mean-proba): {ens_test_loss:.6f} | acc: {ens_test_acc:.6f}")
print("=========================================================\n")


# -------------------------
# 4) Per-run summary (mean ± std) including MSE
# -------------------------
train_loss_mean, train_loss_std = mean_std([r["train_loss"] for r in all_evals])
test_loss_mean,  test_loss_std  = mean_std([r["test_loss"]  for r in all_evals])
train_acc_mean,  train_acc_std  = mean_std([r["train_acc"]  for r in all_evals])
test_acc_mean,   test_acc_std   = mean_std([r["test_acc"]   for r in all_evals])
train_mse_mean,  train_mse_std  = mean_std([r["train_mse"]  for r in all_evals])
test_mse_mean,   test_mse_std   = mean_std([r["test_mse"]   for r in all_evals])

print("\n================ PER-RUN RESULTS ================")
for r in all_evals:
    print(
        f"seed={r['seed']:>2d} | "
        f"train loss={r['train_loss']:.6f}, acc={r['train_acc']:.6f}, mse={r['train_mse']:.6f} | "
        f"test  loss={r['test_loss']:.6f},  acc={r['test_acc']:.6f},  mse={r['test_mse']:.6f}"
    )
print("=================================================\n")

print("================ SUMMARY (mean ± std) ================")
print(f"TRAIN loss: {train_loss_mean:.6f} ± {train_loss_std:.6f}")
print(f"TRAIN acc : {train_acc_mean:.6f} ± {train_acc_std:.6f}")
print(f"TRAIN mse : {train_mse_mean:.6f} ± {train_mse_std:.6f}")
print(f"TEST  loss: {test_loss_mean:.6f} ± {test_loss_std:.6f}")
print(f"TEST  acc : {test_acc_mean:.6f} ± {test_acc_std:.6f}")
print(f"TEST  mse : {test_mse_mean:.6f} ± {test_mse_std:.6f}")
print("======================================================\n")


# -------------------------
# 5) Curves + 3 plots (Loss, Accuracy, MSE)
# -------------------------
train_losses = stack_histories(all_histories, "loss")
test_losses  = stack_histories(all_histories, "val_loss")

train_accs   = stack_histories(all_histories, "Accuracy")
test_accs    = stack_histories(all_histories, "val_Accuracy")

train_mses   = stack_histories(all_histories, "MSE")
test_mses    = stack_histories(all_histories, "val_MSE")

print(f"[{NAME}] Using best_epoch from TRAIN k-fold CV (0-based): {best_epoch_global}\n")

# LOSS
plot_runs_with_mean(
    train_losses, test_losses,
    title=f"{NAME} — Learning Curve — Loss (Train vs Test) — {len(SEEDS)} seeds",
    ylabel="loss",
    best_epoch=best_epoch_global,
    best_label="CV best epoch",
)

# ACC
plot_runs_with_mean(
    train_accs, test_accs,
    title=f"{NAME} — Learning Curve — Accuracy (Train vs Test) — {len(SEEDS)} seeds",
    ylabel="accuracy",
    ylim=(0.5, 1.01),
    best_epoch=best_epoch_global,
    best_label="CV best epoch",
)

# MSE
plot_runs_with_mean(
    train_mses, test_mses,
    title=f"{NAME} — Learning Curve — MSE (Train vs Test) — {len(SEEDS)} seeds",
    ylabel="MSE",
    best_epoch=best_epoch_global,
    best_label="CV best epoch",
)


In [ ]:
# ============================================================
# MONK1 — best_epoch from 5-fold CV (TRAIN ONLY) + multi-seed retrain + Loss/Acc/MSE plots
# ============================================================

NAME = "MONK2"
best_cfg_path = "experiments/MONK/results/monk2/best_config.json"
train_path    = "data/MONK/MONK2/monks-2.train"
test_path     = "data/MONK/MONK2/monks-2.test"

print("\n" + "#" * 90)
print(f"### {NAME} — {len(SEEDS)} seeds + majority vote ensemble")
print("#" * 90)

# Load cfg
if not os.path.exists(best_cfg_path):
    raise FileNotFoundError(f"[{NAME}] best_config not found: {best_cfg_path}")
with open(best_cfg_path, "r") as f:
    cfg_raw = json.load(f)

# The test set is passed to fit() as X_val below purely so the Train-vs-Test
# curves can be logged. Early stopping must therefore be switched off: with
# restore_best_weights it would otherwise pick the epoch that happens to look
# best on the test set, which is model selection on the test set.
cfg = copy.deepcopy(cfg_raw)
cfg.setdefault("callbacks", {})
cfg["callbacks"]["early_stopping"] = False

# Load data
if not os.path.exists(train_path):
    raise FileNotFoundError(f"[{NAME}] Train file not found: {train_path}")
if not os.path.exists(test_path):
    raise FileNotFoundError(f"[{NAME}] Test file not found:  {test_path}")

print("Train file:", train_path)
print("Test  file:", test_path)

X_train, y_train = load_monk(train_path, encode=True)
X_test,  y_test  = load_monk(test_path,  encode=True)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)

# -------------------------
# Training params
# -------------------------
epochs = int(cfg.get("training", {}).get("epochs", 1000))
batch_size = int(cfg.get("training", {}).get("batch_size", 16))
shuffle = bool(cfg.get("training", {}).get("shuffle", True))
drop_last = bool(cfg.get("training", {}).get("drop_last", False))
include_reg_in_val = bool(cfg.get("training", {}).get("include_reg_in_val", False))

# -------------------------
# 1) Best epoch from 5-fold CV on TRAIN only (no test leakage)
#    - compute per seed, then average across seeds
# -------------------------
cv_cfg = cfg.get("cv", {}) or {}
K = int(cv_cfg.get("k", 5))

best_epoch_means = []
best_epoch_stds = []

print("\n" + "#" * 90)
print(f"[{NAME}] Computing best_epoch from {K}-Fold CV on TRAIN ONLY (per seed, then mean)")
print("#" * 90)

for s in SEEDS:
    be_mean, be_std, be_list = compute_best_epoch_from_kfold(
        X_train=X_train,
        y_train=y_train,
        cfg=cfg,
        seed=int(s),
        build_model_from_cfg_fn=build_model_from_cfg,
        trainer_cls=Trainer,
        k=K,
        monitor="val_loss",
    )
    if be_mean is None:
        print(f"  seed={s}: (no best epoch found)")
        continue

    best_epoch_means.append(be_mean)
    best_epoch_stds.append(be_std)
    print(f"  seed={s}: best_epoch_mean={be_mean:.2f} (std across folds={be_std:.2f}) | folds={be_list}")

if len(best_epoch_means) == 0:
    raise RuntimeError("Could not compute best_epoch from k-fold histories (missing 'val_loss'?).")

best_epoch_global = int(np.round(float(np.mean(best_epoch_means))))
print("\n" + "-" * 90)
print(f"[{NAME}] GLOBAL best_epoch (mean across seeds of k-fold mean): {best_epoch_global}  (0-based)")
print("-" * 90 + "\n")


# -------------------------
# 2) Full retraining on TRAIN and evaluation on TEST (one model per seed)
# -------------------------
all_histories, all_models, all_evals, run_lengths = [], [], [], []

print("\n" + "#" * 90)
print(f"[{NAME}] Full retraining for {len(SEEDS)} seeds + logging Train vs Test curves")
print("#" * 90)

for s in SEEDS:
    print("\n" + "=" * 70)
    print(f"[{NAME}] RUN seed={s}")
    print("=" * 70)

    np.random.seed(int(s))

    model = build_model_from_cfg(
        run_cfg=cfg,
        seed=int(s),
        in_dim=X_train.shape[1],
        out_dim=y_train.shape[1],
        task="binary",
    )

    # ---- add MSE metric so History includes MSE / val_MSE
    if not any(m.__class__.__name__ == "MSE" for m in getattr(model, "metrics", [])):
        model.metrics.append(MSEMetric())

    trainer = Trainer(model, verbose=1)

    history = trainer.fit(
        X_train, y_train,
        X_val=X_test, y_val=y_test,   # used ONLY for plotting curves (NOT for best_epoch)
        epochs=epochs,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        seed=int(s),
        include_reg_in_val=include_reg_in_val,
    )

    hist = history.to_dict()
    all_histories.append(hist)
    all_models.append(model)

    L = len(hist.get("loss", []))
    run_lengths.append(L)
    print(f"Done. epochs_ran={L}")

    train_eval = trainer.evaluate(X_train, y_train, batch_size=256, include_regularization=True)
    test_eval  = trainer.evaluate(X_test,  y_test,  batch_size=256, include_regularization=True)

    all_evals.append({
        "seed": int(s),
        "train_loss": float(train_eval.get("loss", np.nan)),
        "train_acc":  float(train_eval.get("Accuracy", np.nan)),
        "train_mse":  float(train_eval.get("MSE", np.nan)),
        "test_loss":  float(test_eval.get("loss", np.nan)),
        "test_acc":   float(test_eval.get("Accuracy", np.nan)),
        "test_mse":   float(test_eval.get("MSE", np.nan)),
    })

print("\nEpochs per run:", run_lengths)
print("Min epochs:", min(run_lengths), "| Max epochs:", max(run_lengths))


# -------------------------
# 3) Majority-vote ensemble (final numbers)
# -------------------------
P_train = np.stack([predict_proba(m, X_train) for m in all_models], axis=0)
P_test  = np.stack([predict_proba(m, X_test)  for m in all_models], axis=0)

ytr = y_train.reshape(-1).astype(int)
yte = y_test.reshape(-1).astype(int)

yhat_tr, pmean_tr, _ = majority_vote(P_train, threshold=0.5)
yhat_te, pmean_te, _ = majority_vote(P_test,  threshold=0.5)

ens_train_acc  = accuracy(ytr, yhat_tr)
ens_test_acc   = accuracy(yte, yhat_te)
ens_train_loss = bce_loss(ytr, pmean_tr)
ens_test_loss  = bce_loss(yte, pmean_te)

print("\n================ ENSEMBLE (MAJORITY VOTE) ================")
print(f"Models: {len(all_models)}")
print(f"TRAIN BCE(mean-proba): {ens_train_loss:.6f} | acc: {ens_train_acc:.6f}")
print(f"TEST  BCE(mean-proba): {ens_test_loss:.6f} | acc: {ens_test_acc:.6f}")
print("=========================================================\n")


# -------------------------
# 4) Per-run summary (mean ± std) including MSE
# -------------------------
train_loss_mean, train_loss_std = mean_std([r["train_loss"] for r in all_evals])
test_loss_mean,  test_loss_std  = mean_std([r["test_loss"]  for r in all_evals])
train_acc_mean,  train_acc_std  = mean_std([r["train_acc"]  for r in all_evals])
test_acc_mean,   test_acc_std   = mean_std([r["test_acc"]   for r in all_evals])
train_mse_mean,  train_mse_std  = mean_std([r["train_mse"]  for r in all_evals])
test_mse_mean,   test_mse_std   = mean_std([r["test_mse"]   for r in all_evals])

print("\n================ PER-RUN RESULTS ================")
for r in all_evals:
    print(
        f"seed={r['seed']:>2d} | "
        f"train loss={r['train_loss']:.6f}, acc={r['train_acc']:.6f}, mse={r['train_mse']:.6f} | "
        f"test  loss={r['test_loss']:.6f},  acc={r['test_acc']:.6f},  mse={r['test_mse']:.6f}"
    )
print("=================================================\n")

print("================ SUMMARY (mean ± std) ================")
print(f"TRAIN loss: {train_loss_mean:.6f} ± {train_loss_std:.6f}")
print(f"TRAIN acc : {train_acc_mean:.6f} ± {train_acc_std:.6f}")
print(f"TRAIN mse : {train_mse_mean:.6f} ± {train_mse_std:.6f}")
print(f"TEST  loss: {test_loss_mean:.6f} ± {test_loss_std:.6f}")
print(f"TEST  acc : {test_acc_mean:.6f} ± {test_acc_std:.6f}")
print(f"TEST  mse : {test_mse_mean:.6f} ± {test_mse_std:.6f}")
print("======================================================\n")


# -------------------------
# 5) Curves + 3 plots (Loss, Accuracy, MSE)
# -------------------------
train_losses = stack_histories(all_histories, "loss")
test_losses  = stack_histories(all_histories, "val_loss")

train_accs   = stack_histories(all_histories, "Accuracy")
test_accs    = stack_histories(all_histories, "val_Accuracy")

train_mses   = stack_histories(all_histories, "MSE")
test_mses    = stack_histories(all_histories, "val_MSE")

print(f"[{NAME}] Using best_epoch from TRAIN k-fold CV (0-based): {best_epoch_global}\n")

# LOSS
plot_runs_with_mean(
    train_losses, test_losses,
    title=f"{NAME} — Learning Curve — Loss (Train vs Test) — {len(SEEDS)} seeds",
    ylabel="loss",
    best_epoch=best_epoch_global,
    best_label="CV best epoch",
)

# ACC
plot_runs_with_mean(
    train_accs, test_accs,
    title=f"{NAME} — Learning Curve — Accuracy (Train vs Test) — {len(SEEDS)} seeds",
    ylabel="accuracy",
    ylim=(0.5, 1.01),
    best_epoch=best_epoch_global,
    best_label="CV best epoch",
)

# MSE
plot_runs_with_mean(
    train_mses, test_mses,
    title=f"{NAME} — Learning Curve — MSE (Train vs Test) — {len(SEEDS)} seeds",
    ylabel="MSE",
    best_epoch=best_epoch_global,
    best_label="CV best epoch",
)


In [ ]:
# ============================================================
# MONK3 — best_epoch from 5-fold CV (TRAIN ONLY) + multi-seed retrain + Loss/Acc/MSE plots
# ============================================================

NAME = "MONK3"
best_cfg_path = "experiments/MONK/results/monk3/best_config.json"
train_path    = "data/MONK/MONK3/monks-3.train"
test_path     = "data/MONK/MONK3/monks-3.test"

print("\n" + "#" * 90)
print(f"### {NAME} — {len(SEEDS)} seeds + majority vote ensemble")
print("#" * 90)

# Load cfg
if not os.path.exists(best_cfg_path):
    raise FileNotFoundError(f"[{NAME}] best_config not found: {best_cfg_path}")
with open(best_cfg_path, "r") as f:
    cfg_raw = json.load(f)

# Load data
if not os.path.exists(train_path):
    raise FileNotFoundError(f"[{NAME}] Train file not found: {train_path}")
if not os.path.exists(test_path):
    raise FileNotFoundError(f"[{NAME}] Test file not found:  {test_path}")

print("Train file:", train_path)
print("Test  file:", test_path)

X_train, y_train = load_monk(train_path, encode=True)
X_test,  y_test  = load_monk(test_path,  encode=True)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)

# -------------------------
# Training params
# -------------------------
cfg = copy.deepcopy(cfg_raw)
cfg.setdefault("callbacks", {})
cfg["callbacks"]["early_stopping"] = False
epochs = int(cfg.get("training", {}).get("epochs", 1000))
batch_size = int(cfg.get("training", {}).get("batch_size", 16))
shuffle = bool(cfg.get("training", {}).get("shuffle", True))
drop_last = bool(cfg.get("training", {}).get("drop_last", False))
include_reg_in_val = bool(cfg.get("training", {}).get("include_reg_in_val", False))

# -------------------------
# 1) Best epoch from 5-fold CV on TRAIN only (no test leakage)
#    - compute per seed, then average across seeds
# -------------------------
cv_cfg = cfg.get("cv", {}) or {}
K = int(cv_cfg.get("k", 5))

best_epoch_means = []
best_epoch_stds = []

print("\n" + "#" * 90)
print(f"[{NAME}] Computing best_epoch from {K}-Fold CV on TRAIN ONLY (per seed, then mean)")
print("#" * 90)

for s in SEEDS:
    be_mean, be_std, be_list = compute_best_epoch_from_kfold(
        X_train=X_train,
        y_train=y_train,
        cfg=cfg,
        seed=int(s),
        build_model_from_cfg_fn=build_model_from_cfg,
        trainer_cls=Trainer,
        k=K,
        monitor="val_loss",
    )
    if be_mean is None:
        print(f"  seed={s}: (no best epoch found)")
        continue

    best_epoch_means.append(be_mean)
    best_epoch_stds.append(be_std)
    print(f"  seed={s}: best_epoch_mean={be_mean:.2f} (std across folds={be_std:.2f}) | folds={be_list}")

if len(best_epoch_means) == 0:
    raise RuntimeError("Could not compute best_epoch from k-fold histories (missing 'val_loss'?).")

best_epoch_global = int(np.round(float(np.mean(best_epoch_means))))  # 0-based epoch index
print("\n" + "-" * 90)
print(f"[{NAME}] GLOBAL best_epoch (mean across seeds of k-fold mean): {best_epoch_global}  (0-based)")
print("-" * 90 + "\n")


# -------------------------
# 2) Full retraining on TRAIN and evaluation on TEST (one model per seed)
# -------------------------
all_histories, all_models, all_evals, run_lengths = [], [], [], []

print("\n" + "#" * 90)
print(f"[{NAME}] Full retraining for {len(SEEDS)} seeds + logging Train vs Test curves")
print("#" * 90)

for s in SEEDS:
    print("\n" + "=" * 70)
    print(f"[{NAME}] RUN seed={s}")
    print("=" * 70)

    np.random.seed(int(s))

    model = build_model_from_cfg(
        run_cfg=cfg,
        seed=int(s),
        in_dim=X_train.shape[1],
        out_dim=y_train.shape[1],
        task="binary",
    )

    # ---- add MSE metric so History includes MSE / val_MSE
    if not any(m.__class__.__name__ == "MSE" for m in getattr(model, "metrics", [])):
        model.metrics.append(MSEMetric())

    trainer = Trainer(model, verbose=1)

    history = trainer.fit(
        X_train, y_train,
        X_val=X_test, y_val=y_test,   # used ONLY for plotting curves (NOT for best_epoch)
        epochs=epochs,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        seed=int(s),
        include_reg_in_val=include_reg_in_val,
    )

    hist = history.to_dict()
    all_histories.append(hist)
    all_models.append(model)

    L = len(hist.get("loss", []))
    run_lengths.append(L)
    print(f"Done. epochs_ran={L}")

    train_eval = trainer.evaluate(X_train, y_train, batch_size=256, include_regularization=True)
    test_eval  = trainer.evaluate(X_test,  y_test,  batch_size=256, include_regularization=True)

    all_evals.append({
        "seed": int(s),
        "train_loss": float(train_eval.get("loss", np.nan)),
        "train_acc":  float(train_eval.get("Accuracy", np.nan)),
        "train_mse":  float(train_eval.get("MSE", np.nan)),
        "test_loss":  float(test_eval.get("loss", np.nan)),
        "test_acc":   float(test_eval.get("Accuracy", np.nan)),
        "test_mse":   float(test_eval.get("MSE", np.nan)),
    })

print("\nEpochs per run:", run_lengths)
print("Min epochs:", min(run_lengths), "| Max epochs:", max(run_lengths))


# -------------------------
# 3) Majority-vote ensemble (final numbers)
# -------------------------
P_train = np.stack([predict_proba(m, X_train) for m in all_models], axis=0)
P_test  = np.stack([predict_proba(m, X_test)  for m in all_models], axis=0)

ytr = y_train.reshape(-1).astype(int)
yte = y_test.reshape(-1).astype(int)

yhat_tr, pmean_tr, _ = majority_vote(P_train, threshold=0.5)
yhat_te, pmean_te, _ = majority_vote(P_test,  threshold=0.5)

ens_train_acc  = accuracy(ytr, yhat_tr)
ens_test_acc   = accuracy(yte, yhat_te)
ens_train_loss = bce_loss(ytr, pmean_tr)
ens_test_loss  = bce_loss(yte, pmean_te)

print("\n================ ENSEMBLE (MAJORITY VOTE) ================")
print(f"Models: {len(all_models)}")
print(f"TRAIN BCE(mean-proba): {ens_train_loss:.6f} | acc: {ens_train_acc:.6f}")
print(f"TEST  BCE(mean-proba): {ens_test_loss:.6f} | acc: {ens_test_acc:.6f}")
print("=========================================================\n")


# -------------------------
# 4) Per-run summary (mean ± std) including MSE
# -------------------------
train_loss_mean, train_loss_std = mean_std([r["train_loss"] for r in all_evals])
test_loss_mean,  test_loss_std  = mean_std([r["test_loss"]  for r in all_evals])
train_acc_mean,  train_acc_std  = mean_std([r["train_acc"]  for r in all_evals])
test_acc_mean,   test_acc_std   = mean_std([r["test_acc"]   for r in all_evals])
train_mse_mean,  train_mse_std  = mean_std([r["train_mse"]  for r in all_evals])
test_mse_mean,   test_mse_std   = mean_std([r["test_mse"]   for r in all_evals])

print("\n================ PER-RUN RESULTS ================")
for r in all_evals:
    print(
        f"seed={r['seed']:>2d} | "
        f"train loss={r['train_loss']:.6f}, acc={r['train_acc']:.6f}, mse={r['train_mse']:.6f} | "
        f"test  loss={r['test_loss']:.6f},  acc={r['test_acc']:.6f},  mse={r['test_mse']:.6f}"
    )
print("=================================================\n")

print("================ SUMMARY (mean ± std) ================")
print(f"TRAIN loss: {train_loss_mean:.6f} ± {train_loss_std:.6f}")
print(f"TRAIN acc : {train_acc_mean:.6f} ± {train_acc_std:.6f}")
print(f"TRAIN mse : {train_mse_mean:.6f} ± {train_mse_std:.6f}")
print(f"TEST  loss: {test_loss_mean:.6f} ± {test_loss_std:.6f}")
print(f"TEST  acc : {test_acc_mean:.6f} ± {test_acc_std:.6f}")
print(f"TEST  mse : {test_mse_mean:.6f} ± {test_mse_std:.6f}")
print("======================================================\n")


# -------------------------
# 5) Curves + 3 plots (Loss, Accuracy, MSE)
# -------------------------
train_losses = stack_histories(all_histories, "loss")
test_losses  = stack_histories(all_histories, "val_loss")

train_accs   = stack_histories(all_histories, "Accuracy")
test_accs    = stack_histories(all_histories, "val_Accuracy")

train_mses   = stack_histories(all_histories, "MSE")
test_mses    = stack_histories(all_histories, "val_MSE")

print(f"[{NAME}] Using best_epoch from TRAIN k-fold CV (0-based): {best_epoch_global}\n")

# LOSS
plot_runs_with_mean(
    train_losses, test_losses,
    title=f"{NAME} — Learning Curve — Loss (Train vs Test) — {len(SEEDS)} seeds",
    ylabel="loss",
    best_epoch=best_epoch_global,
    best_label="CV best epoch",
)

# ACC
plot_runs_with_mean(
    train_accs, test_accs,
    title=f"{NAME} — Learning Curve — Accuracy (Train vs Test) — {len(SEEDS)} seeds",
    ylabel="accuracy",
    ylim=(0.5, 1.01),
    best_epoch=best_epoch_global,
    best_label="CV best epoch",
)

# MSE
plot_runs_with_mean(
    train_mses, test_mses,
    title=f"{NAME} — Learning Curve — MSE (Train vs Test) — {len(SEEDS)} seeds",
    ylabel="MSE",
    best_epoch=best_epoch_global,
    best_label="CV best epoch",
)
